# Analyzing whether each station covers the baseline period and the analysis period and the distribution of datacoverage

## Importing Necessities

In [1]:
import pandas as pd

## Loading the station file

In [3]:
file_path = "../data/raw/louisiana_stations.csv"
stations = pd.read_csv(file_path)

print("Shape:", stations.shape)
stations.head()

Shape: (845, 9)


,elevation,mindate,maxdate,latitude,name,datacoverage,id,elevationUnit,longitude
0,3.7,2012-10-13,2022-01-02,29.97619,"ABBEVILLE 1.0 E, LA US",0.9403,GHCND:US1LAVM0004,METERS,-92.107181
1,3.4,2019-10-03,2026-03-06,29.95550,"ABBEVILLE 1.9 SW, LA US",0.8266,GHCND:US1LAVM0008,METERS,-92.157139
2,2.1,2023-07-04,2024-11-14,29.93645,"ABBEVILLE 10.1 WSW, LA US",0.4940,GHCND:US1LAVM0014,METERS,-92.286200
3,0.9,2019-05-07,2019-10-01,29.87062,"ABBEVILLE 9.9 SW, LA US",0.8108,GHCND:US1LAVM0006,METERS,-92.237521
4,3.0,1891-02-01,2026-03-08,29.96880,"ABBEVILLE, LA US",0.7772,GHCND:USC00160007,METERS,-92.116900


## Inspecting columns and missing values

### Here we can observe that some stations do not provide elevation data

In [5]:
print("Columns:")
print(stations.columns.tolist())

print("\nMissing values:")
print(stations.isna().sum())

Columns:
['elevation', 'mindate', 'maxdate', 'latitude', 'name', 'datacoverage', 'id', 'elevationUnit', 'longitude']

Missing values:
elevation        45
mindate           0
maxdate           0
latitude          0
name              0
datacoverage      0
id                0
elevationUnit    45
longitude         0
dtype: int64


## Converting dates to datetime

In [6]:
stations["mindate"] = pd.to_datetime(stations["mindate"], errors="coerce")
stations["maxdate"] = pd.to_datetime(stations["maxdate"], errors="coerce")

## Basic summary of station metadata

In [7]:
stations[["latitude", "longitude", "datacoverage", "elevation"]].describe()

,latitude,longitude,datacoverage,elevation
count,845.000000,845.000000,845.000000,800.000000
mean,31.063733,-91.877276,0.773311,31.059500
std,1.014487,1.231150,0.256453,29.725935
min,29.016940,-94.000200,0.017600,-1.500000
25%,30.250090,-92.998641,0.637000,5.500000
50%,30.787200,-91.983854,0.886900,21.450000
75%,32.066670,-90.921390,0.975700,52.150000
max,32.992200,-89.170270,1.000000,154.800000


## Defining the two required periods

### Baseline Period: We require only July-August observations from 1981 to 2010 because the heat-wave rule is based on the 85th percentile of historical July-August daily minimum apparent temperature during that baseline period
### Analysis Period: Aligning with the accident data window (Jan 2015 to July 2025)

In [8]:
baseline_start = pd.Timestamp("1981-07-01")
baseline_end = pd.Timestamp("2010-08-31")

analysis_start = pd.Timestamp("2015-01-01")
analysis_end = pd.Timestamp("2025-07-31")

## Checking whether each station covers the baseline period

### 119 Stations have data over the timeperiod required for creating the baseline dataset

In [10]:
stations["covers_baseline"] = (
    (stations["mindate"] <= baseline_start) &
    (stations["maxdate"] >= baseline_end)
)

stations["covers_baseline"].value_counts()

covers_baseline
False    726
True     119
Name: count, dtype: int64

## Checking whether each station covers the analysis period

### 139 Stations have data over the timeperiod required for creating the analysis dataset

In [11]:
stations["covers_analysis"] = (
    (stations["mindate"] <= analysis_start) &
    (stations["maxdate"] >= analysis_end)
)

stations["covers_analysis"].value_counts()

covers_analysis
False    706
True     139
Name: count, dtype: int64

## Checking stations that cover both periods

### 64 stations staisfy the timeperiod requirement for both baseline and analysis dataset

In [12]:
stations["covers_both"] = (
    stations["covers_baseline"] &
    stations["covers_analysis"]
)

stations["covers_both"].value_counts()

covers_both
False    781
True      64
Name: count, dtype: int64

## Datacoverage distribution

### 415 stations have datacoverage >= 0.9. Datacoverage is NOAA’s completeness indicator for how much data an item has relative to what could exist for that query scope

In [13]:
stations["datacoverage"].describe()

count    845.000000
mean       0.773311
std        0.256453
min        0.017600
25%        0.637000
50%        0.886900
75%        0.975700
max        1.000000
Name: datacoverage, dtype: float64

In [14]:
print("Stations with datacoverage >= 0.9:", (stations["datacoverage"] >= 0.9).sum())
print("Stations with datacoverage >= 0.8:", (stations["datacoverage"] >= 0.8).sum())
print("Stations with datacoverage >= 0.7:", (stations["datacoverage"] >= 0.7).sum())

Stations with datacoverage >= 0.9: 415
Stations with datacoverage >= 0.8: 525
Stations with datacoverage >= 0.7: 596


## Strongest candidate stations

In [15]:
candidate_stations = stations[stations["covers_both"]].copy()
candidate_stations = candidate_stations.sort_values("datacoverage", ascending=False)

candidate_stations[["id", "name", "latitude", "longitude", "mindate", "maxdate", "datacoverage"]].head(20)

,id,name,latitude,longitude,mindate,maxdate,datacoverage
704,GHCND:USW00013957,"SHREVEPORT AIRPORT, LA US",32.44730,-93.82440,1939-07-01,2026-03-09,1.0000
65,GHCND:USW00013970,"BATON ROUGE METRO AIRPORT, LA US",30.53782,-91.14681,1930-01-01,2026-03-09,0.9992
424,GHCND:USW00013976,"LAFAYETTE REGIONAL AIRPORT, LA US",30.19859,-91.98957,1893-01-01,2026-03-09,0.9969
313,GHCND:USC00163695,"GONZALES, LA US",30.20330,-90.92250,1978-03-01,2026-03-10,0.9956
169,GHCND:USC00161979,"COLUMBIA LOCK, LA US",32.16730,-92.10769,1972-05-01,2026-03-10,0.9949
83,GHCND:USC00160800,"BIENVILLE 3 NE, LA US",32.37440,-92.94330,1972-11-01,2026-03-09,0.9946
66,GHCND:USC00160558,"BATON ROUGE SHERWOOD, LA US",30.44940,-91.04770,1979-07-01,2025-11-30,0.9943
387,GHCND:USC00164700,"JENNINGS, LA US",30.20020,-92.66410,1897-09-01,2026-03-09,0.9941
622,GHCND:USC00167366,"PLAQUEMINE 2 N, LA US",30.32140,-91.25170,1978-03-01,2026-03-10,0.9939
657,GHCND:USC00167738,"RED RIVER RESEARCH STATION, LA US",32.42190,-93.63800,1975-11-01,2026-03-10,0.9936


## Saving the filtered candidate stations

In [17]:
candidate_stations.to_csv("../data/raw/louisiana_candidate_stations.csv", index=False)
print("Saved candidate station file.")

Saved candidate station file.
